# 02 — Streamflow Model Training

Train a CatchmentLSTM on CAMELS data using the torchharness Trainer.

In [ ]:
from floodrisk.config import ExperimentConfig

cfg = ExperimentConfig.from_yaml("../configs/streamflow/lstm_camels.yaml")
print(cfg)

In [ ]:
from floodrisk.data.camels import CAMELSLoader
from floodrisk.data.normalization import BasinNormalizer

loader = CAMELSLoader(cfg.data.data_dir)
basins = loader.list_basins()[:5]  # use 5 basins for demo

forcing = {b: loader.load_basin_forcing(b, cfg.data.train_start, cfg.data.val_end) for b in basins}
streamflow = {b: loader.load_basin_streamflow(b, cfg.data.train_start, cfg.data.val_end) for b in basins}
print(f"Loaded {len(basins)} basins")

In [ ]:
from floodrisk.datasets.streamflow import CatchmentDataset
from torch.utils.data import DataLoader

train_ds = CatchmentDataset(basins, forcing, streamflow, seq_length=cfg.data.seq_length)
print(f"Training samples: {len(train_ds)}")

train_loader = DataLoader(train_ds, batch_size=cfg.trainer.batch_size, shuffle=True)
val_loader = DataLoader(train_ds, batch_size=cfg.trainer.batch_size)  # same data for demo

In [ ]:
from floodrisk.models import build_model
from floodrisk.losses import NSELoss
from floodrisk.metrics.hydrology import NSEMetric, KGEMetric
from floodrisk.torchharness import Trainer

n_features = train_ds[0][0].shape[-1]
model = build_model(cfg.model.type, n_features=n_features, hidden_size=cfg.model.hidden_size,
                    num_layers=cfg.model.num_layers, dropout=cfg.model.dropout)
loss_fn = NSELoss()
metrics = [NSEMetric(), KGEMetric()]

trainer = Trainer(model, cfg.trainer, train_loader, val_loader, loss_fn, metrics=metrics)
result = trainer.fit()
print(result)